# 第 1 周 第 1 天练习 —— 网页抓取 + OpenRouter 摘要

## 练习目标（理念）

用 **OpenRouter**（OpenAI 兼容 API）做一个「读网页 → 吐槽式短摘要」小工具：

- **输入**：任意网站 URL
- **中间**：`fetch_website_contents` 抓正文
- **输出**：带点 snarky 幽默的 Markdown 摘要（忽略导航类噪音）

作者备注：本练习走 OpenRouter，不直接打官方 OpenAI。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `scraper.fetch_website_contents(url)` |
| Chat Completions | `openrouter.chat.completions.create(...)` |
| `messages`（system / user） | `messages_for(website)` 组装两条消息 |
| 模型路由名 | `openai/gpt-4.1-mini`（OpenRouter 上的模型 id） |
| 笔记本展示 | `display(Markdown(summary))` |

## 怎么跑

1. 同目录需有可用的 `scraper.py`（提供 `fetch_website_contents`）
2. `.env` 里配置 `OPENROUTER_API_BASE_URL` 与 `OPENROUTER_API_KEY`
3. 从上到下运行；最后一格对 `http://juunsdev.my.id` 调用 `display_summary`


### 导入依赖 & 加载环境变量

把标准库、`dotenv`、本地 `scraper`、IPython 展示工具和 OpenAI 客户端一次性引入；随后从 `.env` 读 OpenRouter 配置。


In [ ]:
# ========== 导入 + 读环境：为后面的 OpenRouter 客户端做准备 ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 导入抓取函数：入参 URL，返回网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：把模型返回的 Markdown 漂亮地显示在笔记本里
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：这里用来对接 OpenRouter 的兼容接口
from openai import OpenAI

# 加载 .env；override=True 表示用文件值覆盖已有同名环境变量
load_dotenv(override=True)
# OpenRouter API 根地址（例如 https://openrouter.ai/api/v1）
base_url = os.getenv('OPENROUTER_API_BASE_URL')
# OpenRouter API Key（不要打印）
api_key = os.getenv('OPENROUTER_API_KEY')

# 两个都缺失时打印排查提示；错误文案保持英文原样（可能指向本文件夹 troubleshooting 笔记）
if not base_url and not api_key:
  print("No BASE URL or API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")


### 环境准备：创建 OpenRouter 客户端

用上一格读到的 `base_url` / `api_key` 实例化 `OpenAI(...)`。只要改环境变量，后面业务代码不用动。


In [ ]:
# ========== OpenRouter 客户端：OpenAI SDK + 自定义 base_url ==========

# 创建客户端：请求会发到 OpenRouter，而不是官方 api.openai.com
openrouter = OpenAI(
  base_url=base_url,
  api_key=api_key
)


### 环境准备：messages、摘要与展示函数

定义 system/user 提示词模板，以及 `messages_for` → `summarize` → `display_summary` 三层封装：抓网页、调模型、在笔记本里渲染。


In [ ]:
# ========== Prompt + 业务函数：抓网页 → 组 messages → 调模型 → 展示 ==========

# system prompt 保留英文：定调「吐槽式、幽默、短摘要、忽略导航、直接回 Markdown」
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
# user 前缀保留英文：告诉模型后面跟的是网页正文，并要求顺带总结新闻/公告
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# 把「角色设定 + 网页正文」组装成 Chat Completions 需要的 messages 列表
def messages_for(website):
    return [
        # system：全局行为与语气
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 抓到的网站文本
        {"role": "user", "content": user_prompt_prefix + website}
    ]

# 端到端摘要：URL → 抓取正文 → 调 OpenRouter → 返回助手文本
def summarize(url):
  # 调用本地 scraper：把网页变成纯文本（具体抓取实现见 scraper.py）
  website = fetch_website_contents(url)
  # 非流式 Chat Completions；模型 id 是 OpenRouter 路由名 openai/gpt-4.1-mini
  response = openrouter.chat.completions.create(
      model = "openai/gpt-4.1-mini",
      messages = messages_for(website)
  )
  # 取出第一条 choice 里 assistant 的 content 字段
  return response.choices[0].message.content

# 笔记本友好封装：先 summarize，再用 Markdown 渲染
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


### 抓取并摘要自己的网站

对作者个人站 `http://juunsdev.my.id` 跑一遍完整流水线；可改成任意公开 URL 做对比。


In [ ]:
# ========== 实测：对个人站跑 display_summary ==========

# URL 保持原样；改这里即可换目标网站（需 scraper 能访问）
display_summary("http://juunsdev.my.id")
